# Part 2: Hyperparameter Tunning and Model Evaluation

# Importing necessary libraries

In [79]:
# Import libraries for building and tuning the neural network
from keras.models import Sequential
from keras.layers import Flatten, Dense, Dropout
import keras
import keras_tuner as kt
from keras.utils import to_categorical
from a1 import build_deep_nn

# Load and prepare the MNIST Dataset

In [81]:
# Load MNIST dataset
from keras.datasets import mnist
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

# Normalize the input data to the range[0,1]
train_images = train_images.astype('float32')/255
test_images = test_images.astype('float32')/255

# Reshape images to include the channel dimensions
train_images = train_images.reshape(-1, 28, 28, 1)
test_images = test_images.reshape(-1, 28, 28, 1)

# Convert labels to one-hot encoding for multi-class classification
train_labels = to_categorical(train_labels)
test_labels = to_categorical(test_labels)

# Print shapes to confirm data preparation
# Print the shapes of training and test datasets to ensure that they have been correctly
# processed and reshaped
print("Train set:", train_images.shape)
print("Test set:", test_images.shape)

Train set: (60000, 28, 28, 1)
Test set: (10000, 28, 28, 1)


## Explanation

* The pixel values in the MNIST dataset are in the range[0, 255].
* Dividing by 255 scales these values to the range [0, 1] which is beneficial for neural network training as it helps with numerical stability and gets better results in fewer steps
* Keras expects images in the shape (number of images, height, width, number of color channels). *# We add a channel dimension for grayscale images (set to 1).* # The -1 allows NumPy to automatically figure out the number of images based on the total size and the other dimensions.

# Define the Model Building Function
The below code is essential for defining and configuring a neural network model that will be used to classify images in the MNIST dataset

In [83]:
def build_model(hp):
    # Define hyperparameters for tuning
    num_layers = hp.Int('num_layers', 1, 3, 1)
    hidden_size = hp.Int('hidden_size', 16, 256, 32)
    dropout_rate = hp.Float('dropout_rate', 0.0, 0.8)

    # Initialize the Model
    model = Sequential()
    # Flatten Input Images
    model.add(Flatten(input_shape=(28,28,1)))

    # Add Hidden Layers 
    for i in range(num_layers):
        model.add(Dense(hidden_size, activation = 'relu'))
        if i == num_layers -1:
            model.add(Dropout(dropout_rate))
    # Output Layer
    model.add(Dense(10, activation = 'softmax'))

    # Compile the Model
    model.compile(optimizer = 'adam', 
                  loss = 'categorical_crossentropy',
                  metrics = ['accuracy'])

    return model


# Define the tuner
tuner = kt.BayesianOptimization(
    build_model,
    objective=kt.Objective('val_accuracy', 'max'),
    max_trials=10,
    num_initial_points=2,
    overwrite = True  # Ensures a fresh start each time
)

# Perform hyperparameter search
tuner.search(train_images, train_labels, validation_data=(test_images, test_labels), epochs=5, callbacks=[keras.callbacks.EarlyStopping(patience=1)])

# Retrieving and printing the top 1 hyperparameters and model summary
topN = 1
for x in range(topN):
    print(tuner.get_best_hyperparameters(topN)[x].values)
    print(tuner.get_best_models(topN)[x].summary())

Trial 10 Complete [00h 00m 31s]
val_accuracy: 0.9747999906539917

Best val_accuracy So Far: 0.9793999791145325
Total elapsed time: 00h 03m 48s
{'num_layers': 3, 'hidden_size': 176, 'dropout_rate': 0.491700142480212}


C:\Users\jayar\AppData\Roaming\Python\Python312\site-packages\keras\src\saving\saving_lib.py:576: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 176)            │       138,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 176)            │        31,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 176)            │        31,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 176)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         1,770 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 202,234 (789.98 KB)

 Trainable params: 202,234 (789.98 KB)

 Non-trainable params: 0 (0.00 B)

None


## Explanation
* Created an empty neural network model where layers can be added sequentially.
* Then converts each 2D image (28x28 pixels) into a 1D array (784 pixels). Neural networks require 1D inputs for dense layers
* 
Dense Layers: Each layer performs a linear transformation with a non-linear activation function (relu), enabling the model to learn complex patterns.* 
Dropout: Applied to the last hidden layer to reduce overfitting by randomly setting a fraction of input units to zero during training, improving generalizatio
* The final layer outputs predictions for each class.
* Activation Function (softmax): Converts the raw output scores into probabilities, with each probability representing the likelihood that the input image belongs to a particular class. This function ensures that the sum of all output probabilities is 1, making it suitable for multi-class classification.
* Optimizer (Adam): Efficiently updates model weights with adaptive learning rates
* (Categorical Crossentrop): Assesses how well predicted probabilities match true labels in multi-class classification
*  (Accuray): Measures the proportion of correct predictions to evaluate model performance.n.

# Displaying the Best Model and Hyperparameters

In [85]:
# Get the best hyperparameters
best_hyperparameters = tuner.get_best_hyperparameters(num_trials=1)[0]

# Get the best model
best_model = tuner.get_best_models(num_models=1)[0]

# Print the best hyperparameters
print("Best Hyperparameters:", best_hyperparameters.values)

# Print the model summary
print("Best Model Summary:")
best_model.summary()


Best Hyperparameters: {'num_layers': 3, 'hidden_size': 176, 'dropout_rate': 0.491700142480212}
Best Model Summary:


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 176)            │       138,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 176)            │        31,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 176)            │        31,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 176)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         1,770 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 202,234 (789.98 KB)

 Trainable params: 202,234 (789.98 KB)

 Non-trainable params: 0 (0.00 B)

## Explanation
* Best Hyperparameters: Shows the best settings for the model found during tuning, helping you understand which configuration works best.
* Model Summary: Details the structure of the top-performing model, including its layers and parameters, so you know how the model is built.

# Evaluate the Best Model

In [87]:
# Evaluate the Best Model on the test set
evaluation_results = best_model.evaluate(test_images, test_labels, batch_size=128)
print('Test loss:', evaluation_results[0])
print('Test accuracy:', evaluation_results[1])


79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9762 - loss: 0.0884
Test loss: 0.07407014071941376
Test accuracy: 0.9793999791145325


## What are the hyperparameters of the optimal model?
The hyperparameters of the optimal model are {'num_layers': 3, 'hidden_size': 176, 'dropout_rate': 0.491700142480212}

## What are the accuracy results of the optimal model on the test set?
The accuracy of the optimal model on the test set is approximately 97.94 %.